# eCREAM Analysis 2: Two different populations. Sex and age by cohort

_The categorical look of the live demo, extended with age. Both synthetic datasets. About ten minutes._

**Before you start.** The eCREAM federation holds the UC1 data model (265 variables in four
population blocks) and, today, two **synthetic** datasets of 5,000 subjects, one with dyspnoea
(`DYSP5K`) and one with transient loss of consciousness, TLoC (`TLOC5K`), generated in July 2026
from anonymized mockup samples. No real patient data are involved. Two properties of the
synthetic data shape every result: each cohort was generated from its own sample, so contrasts
*between* the datasets survive; each block of variables was resampled on its own, so
associations *across* blocks (age and death, saturation and death) do not. Known artefacts:
uniform triage codes and modes of arrival, door-to-nurse at zero, no "discharged" exit, ages
above 89 not capped.

**How this notebook works.** Every analysis is a *federated* call: the code selects the datasets
and the variables, defines a cohort with filters and asks the MIP for aggregates (counts, means,
a test, a model). No record ever reaches this notebook. Run the cells in order; each step says
what to look for and what result to expect. The same steps, for the web interface, are in
`docs/teaching_scenarios/`.

**What this analysis shows.** The two arms of UC1 are not two samples of one population: acute
breathlessness skews male because of what causes it (COPD, heart failure, a smoking history);
fainting is spread evenly across the sexes, with its own age profile. Before any comparison of
the cohorts, or any model of an outcome across them, the demographics have to be looked at,
which is why the UC1 outcome model carries sex and age as covariates. The question: **do the two
cohorts differ in sex and in age?**

## 1. Connect to the eCREAM federation and open the UC1 pathology

In [ ]:
import mip
from mip.filters import F

client = mip.Client.from_env()          # connection configured by the MIP portal
catalog = client.catalog()

try:
    dm = catalog.data_model("UC1")         # add version="eCREAMv8" if the catalogue holds several versions
except LookupError:
    display(catalog.summaries())           # the pathology label differs: pick it from this list
    raise
def dataset(code, label):
    # by the code, which is how the federation labels its datasets (seen 2026-10-12), else by the long label
    try:
        return dm.datasets[code]
    except (KeyError, LookupError):
        return dm.datasets[label]

dysp = dataset("DYSP5K", "Synthetic Dyspnea dataset (5000 subjects)")
tloc = dataset("TLOC5K", "Synthetic TLoC dataset (5000 subjects)")
DATASETS = [dysp, tloc]                   # every call of this notebook runs on both synthetic datasets
display(dm)
display(dm.datasets.to_frame())           # 21 dataset codes: four hold data today, 17 are the EDs' containers

In [ ]:
# The variable tree: the four population blocks of the UC1 case report form. The demo uses
# ALL_PATIENTS (logistics, demographics, door-to-event delays, dataset of origin) and, for the
# mortality model, two variables of DYSPNEA_AND_TLOC (vital signs, follow-up).
dm.tree(group="ALL_PATIENTS", include_variables=True)

## 2. Toolbox

In [ ]:
# ---- Toolbox: small helpers that turn MIP results into readable tables and charts. ----
# Run this cell once. You do not need to read it to follow the analysis; come back to it
# when you want to see how a result payload is unpacked.
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from mip.preprocessing import MissingValuesHandler

POOLED = "all datasets"                 # the row describe() adds across the selected datasets
PCT = "{:.1f}"                          # number format of the percentage columns


def analysis(variables, filters=None, drop_missing=False, datasets=None):
    """A Pipeline on the datasets of this notebook (DATASETS, or the ones given) with the given
    variables and cohort filter. Tests and models need drop_missing=True: the platform requires an
    explicit missing-values step and analyses complete cases only. describe() keeps it off so it
    can count missing values."""
    handler = MissingValuesHandler(strategies={v: "drop" for v in variables}) if drop_missing else None
    return mip.Pipeline(analysis_set=dm.select(datasets=datasets or DATASETS, variables=variables),
                        filters=filters, handle_missing=handler)


def labels_of(variable):
    """{code: label} of a categorical variable, whatever format the catalogue uses."""
    raw = variable._data.get("enumerations") or {}
    if isinstance(raw, dict):
        return {str(k): str(v) for k, v in raw.items()}
    return {str(item.get("code") if isinstance(item, dict) else item):
            str(item.get("label") if isinstance(item, dict) else item) for item in raw}


def name_of(dataset):
    """Column name of a dataset in the tables: its code, or 'all datasets' for the pooled row."""
    return POOLED if dataset == POOLED else dataset._code


def rows_for(result, variable, dataset=POOLED):
    """The describe row of one variable: the pooled row by default, or the row of one dataset."""
    keys = (POOLED,) if dataset == POOLED else (dataset._code, dataset.label)
    return [r for r in result.raw.get("featurewise", [])
            if r.get("variable") == variable._code and r.get("dataset") in keys]


def numeric_by_dataset(result, variable, datasets=None):
    """n, missing, mean, SD, min, quartiles and max of a numerical variable, one row per dataset
    and the pooled row, from a describe result."""
    rows = []
    for ds in list(datasets or DATASETS) + [POOLED]:
        for r in rows_for(result, variable, ds):
            d = r.get("data") or {}
            rows.append({"dataset": name_of(ds), "n": d.get("num_dtps"), "missing": d.get("num_na"),
                         "mean": d.get("mean"), "sd": d.get("std"), "min": d.get("min"), "q1": d.get("q1"),
                         "median": d.get("q2"), "q3": d.get("q3"), "max": d.get("max")})
    return pd.DataFrame(rows)


def counts_by_dataset(result, variable, datasets=None):
    """Counts and percentages of the levels of a categorical variable, one column pair per dataset
    and for the pooled row; the number of missing values in .attrs['missing']."""
    lab = labels_of(variable)
    table, missing = {}, {}
    for ds in list(datasets or DATASETS) + [POOLED]:
        for r in rows_for(result, variable, ds):
            d = r.get("data") or {}
            counts = {str(k): v for k, v in (d.get("counts") or {}).items()}
            total = sum(counts.values())
            table[name_of(ds)] = pd.Series({label: counts.get(c, 0) for c, label in lab.items()})
            table[name_of(ds) + " %"] = pd.Series({label: 100 * counts.get(c, 0) / total if total else np.nan
                                                   for c, label in lab.items()})
            missing[name_of(ds)] = d.get("num_na")
    out = pd.DataFrame(table)
    out.index.name = variable.label
    out.attrs["missing"] = missing
    return out


def record_counts(filters, variable, datasets=None):
    """Records matching a filter, per dataset and pooled, from a describe of one variable that is
    never missing (the dataset of origin): the notebook's version of the filter counter."""
    res = analysis([variable], filters, datasets=datasets).describe()
    out = {}
    for ds in list(datasets or DATASETS) + [POOLED]:
        rows = rows_for(res, variable, ds)
        d = (rows[0].get("data") or {}) if rows else {}
        out[name_of(ds)] = d.get("num_total")          # None when fewer than 10 records match
    return out


def share_of(variable, level, filters, datasets=None):
    """% of the records with a given level (code) of a categorical variable among the records
    where it is recorded, pooled over the datasets, for a cohort filter."""
    res = analysis([variable], filters, datasets=datasets).describe()
    rows = rows_for(res, variable)
    counts = ((rows[0].get("data") or {}).get("counts") or {}) if rows else {}
    total = sum(counts.values())
    return (100 * counts.get(level, 0) / total if total else np.nan), total


def histograms_by_dataset(variable, bins=20, filters=None, datasets=None, unit=""):
    """One histogram per dataset (one federated call each), drawn side by side. Bins with fewer
    than 10 records come back empty: the platform's privacy rule."""
    datasets = list(datasets or DATASETS)
    fig, axes = plt.subplots(1, len(datasets), figsize=(6 * len(datasets), 3.4))
    for ax, ds in zip(np.atleast_1d(axes), datasets):
        h = analysis([variable], filters, drop_missing=True, datasets=[ds]).histogram(variable=variable, bins=bins).raw["histogram"][0]
        edges, counts = h["bins"], h["counts"]
        bars = [c or 0 for c in counts]
        ax.bar(edges[:-1], bars, width=np.diff(edges), align="edge", color="#4C72B0")
        hidden = sum(c is None for c in counts)
        peak = int(np.argmax(bars))
        ax.set_title(f"{variable.label}, {ds.label}")
        ax.set_xlabel(f"{variable.label} ({unit})" if unit else variable.label)
        ax.set_ylabel("records")
        print(f"{ds._code}: bins {edges[1] - edges[0]:.1f} wide, peak {bars[peak]} records at "
              f"{edges[peak]:.0f} to {edges[peak + 1]:.0f}, {hidden} bins under 10 records not drawn")
    plt.tight_layout()
    plt.show()


def fmt_p(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return "n/a"
    return "< 0.001" if p < 0.001 else f"{p:.3f}"


def ttest_summary(result):
    """Independent t-test result -> t, p, df, mean difference with its 95% CI, Cohen's d, n."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    return {k: raw.get(k) for k in ("t_stat", "p", "df", "mean_diff", "ci_lower", "ci_upper", "cohens_d", "n_obs")}


def show_ttest(result, what, unit=""):
    # one line with the t-test result; a key the platform does not return is shown as n/a
    s = ttest_summary(result)
    num = lambda v, spec: format(v, spec) if isinstance(v, (int, float)) else "n/a"
    print(f"{what}: difference of means {num(s['mean_diff'], '.1f')} {unit} (95% CI {num(s['ci_lower'], '.1f')} "
          f"to {num(s['ci_upper'], '.1f')}), t = {num(s['t_stat'], '.1f')} on {num(s['df'], '.0f')} df, "
          f"p = {fmt_p(s['p'])} ({num(s['p'], '.1e')}), Cohen's d {num(s['cohens_d'], '.2f')}, n = {num(s['n_obs'], ',')}")
    missing = [k for k, v in s.items() if v is None]
    if missing:
        print(f"   (keys the platform did not return: {', '.join(missing)}; keys returned: {', '.join(result.raw)})")
    return s


def chi2_summary(result, x, y):
    """Chi-squared Test result -> chi2, p, dof, and the expected counts as a table with labels."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    lab_x, lab_y = labels_of(x), labels_of(y)
    expected = pd.DataFrame(raw.get("expected") or [],
                            index=[lab_x.get(str(i), str(i)) for i in raw.get("x_labels") or []],
                            columns=[lab_y.get(str(c), str(c)) for c in raw.get("y_labels") or []])
    expected.index.name, expected.columns.name = x.label, y.label
    return raw.get("chi2"), raw.get("p_value", raw.get("p")), raw.get("dof"), expected


def term_label(term, variables):
    """Model term -> readable label: 'sex[2]' -> 'Sex = Female', 'age' -> 'Age (per unit)'."""
    by_code = {v._code: v for v in variables}
    term = str(term)
    if "[" in term and term.endswith("]"):
        base, level = term[:-1].split("[", 1)
        v = by_code.get(base)
        return f"{v.label} = {labels_of(v).get(level, level)}" if v else term
    v = by_code.get(term)
    return f"{v.label} (per unit)" if v else term


def odds_ratio_table(result, variables):
    """Logistic regression result -> one row per term with coefficient, odds ratio, 95% CI and p.
    Categorical covariates appear as 'variable = level' relative to the reference level
    (the first code of the variable) that the platform leaves out."""
    frame = result.to_frame().copy()
    frame = frame[~frame["feature"].astype(str).str.lower().isin(["intercept", "const"])]
    frame["term"] = frame["feature"].map(lambda t: term_label(t, variables))
    for col in ("coefficient", "odds_ratio", "or_lower_ci", "or_upper_ci", "p"):
        if col not in frame.columns:
            frame[col] = np.nan
    out = frame[["term", "coefficient", "odds_ratio", "or_lower_ci", "or_upper_ci", "p"]]
    return out.rename(columns={"or_lower_ci": "ci_low", "or_upper_ci": "ci_high"}).reset_index(drop=True)


def show_odds_ratios(table, title):
    """Table and forest plot (log scale) of odds ratios."""
    display(table.style.format({"coefficient": "{:.4f}", "odds_ratio": "{:.3f}", "ci_low": "{:.3f}",
                                "ci_high": "{:.3f}", "p": "{:.3f}"}).hide(axis="index"))
    t = table.iloc[::-1]
    fig, ax = plt.subplots(figsize=(7, 0.6 * len(t) + 1.5))
    ax.errorbar(t["odds_ratio"], range(len(t)),
                xerr=[t["odds_ratio"] - t["ci_low"], t["ci_high"] - t["odds_ratio"]], fmt="o", capsize=4)
    ax.axvline(1, linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ax.set_yticks(range(len(t)))
    ax.set_yticklabels(t["term"])
    ax.set_xlabel("odds ratio (log scale)")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def roc_and_matrix(cv, event="death", non_event="survivor"):
    """Cross-validation result -> the ROC curve of every fold with its area, and the confusion
    matrix summed over the folds at the platform's 50% cut-off."""
    curves, cm = cv.raw["roc_curves"], cv.raw["confusion_matrix"]
    aucs = [c["auc"] for c in curves]
    print("Area under the ROC curve, per fold:", ", ".join(f"{a:.2f}" for a in aucs), f"| average {np.mean(aucs):.2f}")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
    for c in curves:
        axes[0].plot(c["fpr"], c["tpr"], linewidth=1, label=f"{c['name'].replace('_', ' ')}: area {c['auc']:.2f}")
    axes[0].plot([0, 1], [0, 1], linestyle="--", color="grey", label="coin toss: area 0.50")
    axes[0].set_xlabel(f"share of {non_event}s wrongly called {event}s")
    axes[0].set_ylabel(f"share of {event}s caught")
    axes[0].set_title("ROC curve, on patients the model had not seen")
    axes[0].legend(fontsize=8, loc="lower right")
    matrix = np.array([[cm["tn"], cm["fp"]], [cm["fn"], cm["tp"]]])  # rows: the truth; columns: the verdict
    axes[1].imshow(matrix, cmap="Blues")
    for i in range(2):
        for j in range(2):
            axes[1].text(j, i, f"{matrix[i, j]:,}", ha="center", va="center", fontsize=14,
                         color="white" if matrix[i, j] > matrix.max() / 2 else "black")
    axes[1].set_xticks([0, 1])
    axes[1].set_xticklabels([f"called a {non_event}", f"called a {event}"])
    axes[1].set_yticks([0, 1])
    axes[1].set_yticklabels([f"was a {non_event}", f"was a {event}"])
    axes[1].set_title("Confusion matrix at the platform's 50% cut-off")
    plt.tight_layout()
    plt.show()
    n = sum(cm.values())
    sens = 100 * cm["tp"] / (cm["tp"] + cm["fn"]) if cm["tp"] + cm["fn"] else float("nan")
    spec = 100 * cm["tn"] / (cm["tn"] + cm["fp"]) if cm["tn"] + cm["fp"] else float("nan")
    print(f"Accuracy {100 * (cm['tp'] + cm['tn']) / n:.1f}%, sensitivity {sens:.1f}%, specificity {spec:.1f}%")
    return aucs


def saved_experiments():
    """The experiments saved from the web interface (My experiments), as the platform lists them."""
    items = client.experiments().list()
    if not items:
        print("No saved experiment is listed (nothing saved yet, or the registry answers differently on this platform).")
        return pd.DataFrame()
    return pd.DataFrame(items)


print("Toolbox ready.")

## 3. The datasets and the variables of this analysis

| Code | Label in the MIP | Tree location (under UC1) | Role | Values |
|---|---|---|---|---|
| `dataset` | Dataset Registry origin | ALL_PATIENTS / Dataset Registry Origin | which cohort a record belongs to; predictor of the chi-square, grouping of the t-test | 21 codes, listed in the tree |
| `sex` | Sex | ALL_PATIENTS / Demographics | outcome of the Chi-squared Test | 1 Male; 2 Female; 3 Unknown |
| `age` | Age | ALL_PATIENTS / Demographics | outcome of the Independent t-test | integer |

Variables are picked by their **label**, as the tree shows it; the tree location tells you where
to find them in `dm.tree()` and in the MIP interface. The code (for example `door_doctor_delay`)
is what the csv column is called; you will see it in some result payloads. Both synthetic
datasets are selected for every call.

In [ ]:
sex        = dm.variables["Sex"]
age        = dm.variables["Age"]
dataset    = dm.variables["Dataset Registry origin"]      # which cohort a record belongs to
display(mip.to_frame([sex, age, dataset]))

## Step 1. Sex by cohort, descriptively

*Expected:* dyspnoea 3,390 men (67.8%), 1,593 women (31.9%), 17 unknown (0.3%); TLoC 2,419
(48.4%), 2,569 (51.4%), 12 (0.2%); all datasets 5,809, 4,162, 29. Of the records with a known
sex, 68.0% are men in the dyspnoea cohort and 48.5% in the TLoC cohort.

In [ ]:
review = analysis([sex, age]).describe()
by_sex = counts_by_dataset(review, sex)
display(by_sex.style.format({c: PCT for c in by_sex.columns if c.endswith("%")}))
print("missing:", by_sex.attrs["missing"])
ax = by_sex[[c for c in by_sex.columns if c.endswith("%") and not c.startswith(POOLED)]].T.plot(kind="bar", stacked=True, figsize=(5, 3.2), rot=0, color=["#4C72B0", "#DD8452", "#999999"])
ax.set_ylabel("% of records"); ax.set_title("Sex by cohort"); ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

## Step 2. The Chi-squared Test, with and without the "Unknown" code

Outcome sex, predictor the dataset of origin. Without a filter the table is 3 by 2 because
"Unknown" is a level like any other; with the filter `Sex is Male or Female` it is 2 by 2 and
the platform applies Yates' correction.

*Expected:* without a filter chi-square 392.0 on 2 degrees of freedom, p of the order of 10^-86,
with expected counts of about 2,905 men per cohort under independence; with the filter
chi-square 390.4, p of the order of 10^-87 (the figure of the demo slides). Either way the sex
ratio differs beyond any doubt; the filtered version is the one to quote, because "Unknown" is
not a sex.

In [ ]:
chi_all = analysis([sex, dataset], drop_missing=True).chi_square_test(x=dataset, y=sex)
chi2, p, dof, expected = chi2_summary(chi_all, dataset, sex)
print(f"No filter: chi2 = {chi2:.1f} on {dof} df, p = {fmt_p(p)} ({p:.1e})")
display(expected.round(1))

known_sex = F(sex).isin(["1", "2"])
chi_known = analysis([sex, dataset], known_sex, drop_missing=True).chi_square_test(x=dataset, y=sex)
chi2, p, dof, expected = chi2_summary(chi_known, dataset, sex)
print(f"Sex is Male or Female: chi2 = {chi2:.1f} on {dof} df (Yates' correction), p = {fmt_p(p)} ({p:.1e})")
display(expected.round(1))

## Step 3. Age by cohort

The describe of Step 1 already holds the age; then the counter in three age bands, and the
Independent t-test of age by dataset.

*Expected:* dyspnoea mean 69.4 years (SD 19.7), median 72 (quartiles 56 and 85), 18 to 121;
TLoC 66.6 (21.6), median 70 (51 and 83), 18 to 123; all datasets 68.0, median 71. Under 65 /
65 to 79 / 80 and over: 36.4 / 28.2 / 35.4% of the dyspnoea cohort, 42.1 / 25.8 / 32.1% of the
TLoC cohort; over 89: 795 and 776 records (16% of each). t-test: difference 2.85 years (95% CI
2.0 to 3.7), t = 6.9 on 9,998 df, p < 0.001, Cohen's d 0.14. The ages above 89 are an artefact
of the generator's jitter: the DTA requires age over 89 to be delivered as "90+".

In [ ]:
display(numeric_by_dataset(review, age).style.format({c: "{:.1f}" for c in ("mean", "sd")}).hide(axis="index"))
total = record_counts(None, dataset)
rows = []
for label, f in (("under 65", F(age) < 65), ("65 to 79", (F(age) >= 65) & (F(age) < 80)), ("80 and over", F(age) >= 80), ("over 89", F(age) > 89)):
    n = record_counts(f, dataset)
    rows.append({"age": label, **{k: n[k] for k in n}, **{f"{k} %": (100 * n[k] / total[k] if n[k] is not None else np.nan) for k in n}})
bands = pd.DataFrame(rows)
display(bands.style.format({c: PCT for c in bands.columns if c.endswith("%")}).hide(axis="index"))
histograms_by_dataset(age, bins=20, unit="years")
t = analysis([age, dataset], drop_missing=True).t_test(variable=age, group_by=dataset, group_a="DYSP5K", group_b="TLOC5K")
show_ttest(t, "Age, dyspnoea minus TLoC", "years")

## Step 4. State the result

A model answer:

> The two synthetic cohorts differ in sex and age: 68% of the dyspnoea patients are men against
> 48% of the TLoC patients (chi-square 390, p < 0.001), and the dyspnoea patients are three
> years older on average (69.4 against 66.6 years, p < 0.001). Any comparison of the two
> cohorts, and any model of an outcome across them, must adjust for sex and age, which is what
> the UC1 outcome model does.